In [1]:
%reload_ext autoreload
%autoreload 1

# Example Naive Forecast Submission

- Naive (use last observed training value for all future predictions)
- Score using `wrmsse` across all hierarchies

In [2]:
import os
import time
from loguru import logger
from pathlib import Path

start_time = time.time()

# Read Data

In [3]:
from src.process import *
from datasetsforecast.m5 import M5Evaluation

HORIZON = 28
fcols = [f"F{i}" for i in range(1, HORIZON + 1)]

TRAIN_START = 1
TRAIN_END = 1941

TEST_START = 1942
TEST_END = 1969

sales = load_sales(PATH_INPUT, load_prices(PATH_INPUT))
naive = sales.set_index("id")[f"d_{TRAIN_END}"]   # last observed
truth = pd.read_csv(PATH_INPUT / "sales_test_evaluation.csv")

2026-03-23 04:27:05.199 | DEBUG    | src.process:load_prices:44 - Begin Loading Price Data...
2026-03-23 04:27:06.418 | DEBUG    | src.process:load_sales:59 - Begin Loading Sales Data...


# Create Naive Submission

In [4]:
TEST_DAYS = [f"d_{x}" for x in range(TEST_START, TEST_END + 1)]

truth = pd.read_csv(PATH_INPUT / "sales_test_evaluation.csv")

naive_submission = truth.copy()

for test_day in TEST_DAYS:
    naive_submission[test_day] = naive.values

naive_submission["unique_id"] = naive_submission["item_id"] + "_" + naive_submission["store_id"]

In [5]:
naive_submission[TEST_DAYS]

,d_1942,d_1943,d_1944,d_1945,d_1946,d_1947,d_1948,d_1949,d_1950,d_1951,...,d_1960,d_1961,d_1962,d_1963,d_1964,d_1965,d_1966,d_1967,d_1968,d_1969
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,...,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0,6.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30485,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
30486,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30487,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
30488,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
naive_score = M5Evaluation.evaluate("data", naive_submission)

In [7]:
naive_score

,wrmsse
Total,1.752010
Level1,1.966796
Level2,1.904404
Level3,1.879622
Level4,1.946932
Level5,1.913625
Level6,1.880508
Level7,1.878335
Level8,1.798140
Level9,1.764153


In [8]:
naive_submission.to_parquet("data/naive_benchmark.snap.parquet")

In [9]:
logger.info(f"""Naive Score: {naive_score["wrmsse"].mean().mean():.3f}""")
logger.info(f"""Notebook finished in {(time.time() - start_time) / 60:.2f}m""")

2026-03-23 04:27:19.679 | INFO     | __main__:<module>:1 - Naive Score: 1.752
2026-03-23 04:27:19.681 | INFO     | __main__:<module>:2 - Notebook finished in 0.27m
